# Unfiltered IG Masking Sensitivity Analysis

This notebook runs the supplementary unfiltered sensitivity analysis for the IG masking faithfulness experiment over all 61 splitB allergenic proteins.


In [1]:
RUN_TARGET = "colab"  # "colab" | "local"


In [2]:
if RUN_TARGET == "colab":
    import importlib.metadata as _md
    import subprocess as _sp
    import sys as _sys

    _required = {
        "numpy": "1.26.4",
        "scipy": "1.15.3",
        "pandas": "2.3.2",
        "scikit-learn": "1.8.0",
        "transformers": "4.48.1",
        "huggingface-hub": "0.36.2",
        "statsmodels": "0.14.5",
        "seaborn": "0.13.2",
        "matplotlib": "3.10.6",
    }

    def _version_matches(name: str, expected: str) -> bool:
        try:
            return _md.version(name) == expected
        except _md.PackageNotFoundError:
            return False

    _missing_or_mismatched = [
        f"{name}=={version}"
        for name, version in _required.items()
        if not _version_matches(name, version)
    ]

    if _missing_or_mismatched:
        print("Installing:", ", ".join(_missing_or_mismatched))
        _sp.run([
            _sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            *_missing_or_mismatched,
        ], check=True)
        raise SystemExit(
            "Colab environment updated. Restart the runtime once, then rerun the notebook from the top."
        )
    else:
        print("Colab environment already compatible. No reinstall needed.")
else:
    print("Local environment detected. Skipping Colab setup.")


Colab environment already compatible. No reinstall needed.


In [3]:
if RUN_TARGET == "colab":
    from google.colab import drive as _drive
    from pathlib import Path

    _drive.mount("/content/drive", force_remount=False)

    DRIVE_ROOT = Path("/content/drive/MyDrive/XAllergen2.0")
    DRIVE_MODELS = DRIVE_ROOT / "models"
    DRIVE_RESULTS = DRIVE_ROOT / "results"
    DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

    print(f"Google Drive mounted: {DRIVE_ROOT}")
    print(f"Models will sync to: {DRIVE_MODELS}")
    print(f"Results will sync to: {DRIVE_RESULTS}")
else:
    print("Local run: skipping Google Drive mount.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted: /content/drive/MyDrive/XAllergen2.0
Models will sync to: /content/drive/MyDrive/XAllergen2.0/models
Results will sync to: /content/drive/MyDrive/XAllergen2.0/results


In [4]:
import os
import shutil as _shutil
import sys
import urllib.request as _urlreq
from pathlib import Path

from huggingface_hub import snapshot_download

RAW = "https://raw.githubusercontent.com/Jeffateth/XAllergen2.0/main"

if RUN_TARGET == "colab":
    RUNTIME_ROOT = Path("/content/XAllergen2.0")
    _SRC_DIR = RUNTIME_ROOT / "src"
    _PACKAGE_DIR = _SRC_DIR / "xallergen"
    _DATA_DIR = RUNTIME_ROOT / "data"
    _MODEL_DIR = RUNTIME_ROOT / "models"
    _RESULTS_DIR = RUNTIME_ROOT / "results"
    _PROBING_ROWS_DIR = _RESULTS_DIR / "probing" / "rows"
    _FRESH_ESM2_DIR = RUNTIME_ROOT / "hf_models" / "facebook_esm2_t6_8M_UR50D"
    for _d in [_PACKAGE_DIR, _DATA_DIR, _MODEL_DIR, _PROBING_ROWS_DIR, _FRESH_ESM2_DIR]:
        _d.mkdir(parents=True, exist_ok=True)

    for _module_name in [
        "__init__.py",
        "baseline_notebook_utils.py",
        "mtl_epitope_notebook_utils.py",
    ]:
        _drive_module = DRIVE_ROOT / "src" / "xallergen" / _module_name
        _target = _PACKAGE_DIR / _module_name
        if _drive_module.exists():
            _shutil.copy2(_drive_module, _target)
            print(f"Copied from Drive src: {_module_name}")
        else:
            _urlreq.urlretrieve(f"{RAW}/src/xallergen/{_module_name}", _target)
            print(f"Downloaded: {_module_name}")

    for _fname in ["positives_splitB.csv"]:
        _drive_path = DRIVE_ROOT / "data" / _fname
        _target = _DATA_DIR / _fname
        if _drive_path.exists():
            _shutil.copy2(_drive_path, _target)
            print(f"Copied data from Drive: {_fname}")
        else:
            _urlreq.urlretrieve(f"{RAW}/data/{_fname}", _target)
            print(f"Downloaded data: {_fname}")

    _checkpoint_name = "baseline_frozen_esm2.pt"
    _drive_checkpoint = DRIVE_MODELS / _checkpoint_name
    _runtime_checkpoint = _MODEL_DIR / _checkpoint_name
    if _drive_checkpoint.exists():
        _shutil.copy2(_drive_checkpoint, _runtime_checkpoint)
        print(f"Copied checkpoint from Drive: {_drive_checkpoint}")
    else:
        _urlreq.urlretrieve(f"{RAW}/models/{_checkpoint_name}", _runtime_checkpoint)
        print(f"Downloaded checkpoint from GitHub: {_checkpoint_name}")

    _raw_ig_name = "baseline_probing_rows.csv"
    _drive_raw_ig = DRIVE_RESULTS / "probing" / "rows" / _raw_ig_name
    _runtime_raw_ig = _PROBING_ROWS_DIR / _raw_ig_name
    if _drive_raw_ig.exists():
        _shutil.copy2(_drive_raw_ig, _runtime_raw_ig)
        print(f"Copied raw IG artifact from Drive: {_drive_raw_ig}")
    else:
        _urlreq.urlretrieve(f"{RAW}/results/probing/rows/{_raw_ig_name}", _runtime_raw_ig)
        print(f"Downloaded raw IG artifact: {_raw_ig_name}")

    _drive_hf_dir = DRIVE_ROOT / "hf_models" / "facebook_esm2_t6_8M_UR50D"
    if _drive_hf_dir.exists():
        os.environ["XALLERGEN_HF_MODEL_DIR"] = str(_drive_hf_dir)
        print(f"Using cached ESM-2 assets from Drive: {_drive_hf_dir}")
    elif _FRESH_ESM2_DIR.exists() and any(_FRESH_ESM2_DIR.iterdir()):
        os.environ["XALLERGEN_HF_MODEL_DIR"] = str(_FRESH_ESM2_DIR)
        print(f"Using cached ESM-2 assets from runtime: {_FRESH_ESM2_DIR}")
    else:
        _cached_path = snapshot_download(
            repo_id="facebook/esm2_t6_8M_UR50D",
            local_dir=_FRESH_ESM2_DIR,
            local_dir_use_symlinks=False,
        )
        os.environ["XALLERGEN_HF_MODEL_DIR"] = str(_cached_path)
        print(f"Fetched ESM-2 assets required by the trained baseline checkpoint: {_cached_path}")

    if str(RUNTIME_ROOT) not in sys.path:
        sys.path.insert(0, str(RUNTIME_ROOT))
    if str(_SRC_DIR) not in sys.path:
        sys.path.insert(0, str(_SRC_DIR))
else:
    for _candidate in [Path.cwd(), *Path.cwd().parents]:
        _src_dir = _candidate / "src"
        if (_src_dir / "xallergen").exists() and str(_src_dir) not in sys.path:
            sys.path.insert(0, str(_src_dir))
            RUNTIME_ROOT = _candidate
            break


Downloaded: __init__.py
Downloaded: baseline_notebook_utils.py
Downloaded: mtl_epitope_notebook_utils.py
Copied data from Drive: positives_splitB.csv
Copied checkpoint from Drive: /content/drive/MyDrive/XAllergen2.0/models/baseline_frozen_esm2.pt
Copied raw IG artifact from Drive: /content/drive/MyDrive/XAllergen2.0/results/probing/rows/baseline_probing_rows.csv


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

tf_model.h5:   0%|          | 0.00/30.3M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Fetched ESM-2 assets required by the trained baseline checkpoint: /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D


In [5]:
import ast
import importlib
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon

import xallergen.baseline_notebook_utils as baseline_notebook_utils
import xallergen.mtl_epitope_notebook_utils as mtl_epitope_notebook_utils

importlib.reload(baseline_notebook_utils)
importlib.reload(mtl_epitope_notebook_utils)

from xallergen.baseline_notebook_utils import (
    build_tokenizer,
    configure_matplotlib_cache,
    detect_device,
    find_project_root,
    load_baseline_checkpoint,
    parse_epitope_label,
    print_runtime_context,
    validate_ig_residues_by_masking,
)
from xallergen.mtl_epitope_notebook_utils import bootstrap_mean_ci

if RUN_TARGET == "colab":
    import matplotlib
    matplotlib.use("Agg")
else:
    configure_matplotlib_cache(Path.cwd())


In [14]:
DEFAULT_K_PERCENTAGES = (0.05, 0.10, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50)
DEFAULT_MIN_DELTA_P = 0.05
DEFAULT_CONFIDENCE_THRESHOLD = 0.70
DEFAULT_SELECTED_K_PCT = 0.40
DEFAULT_N_RANDOM_TRIALS = 100
RAW_IG_SCORE_COLUMN_CANDIDATES = (
    "ig_scores",
    "ig_scores_json",
    "integrated_gradients_scores",
    "integrated_gradients_scores_json",
)


def load_splitb_positive_sequence_lookup(data_dir: Path):
    positive_test_df = pd.read_csv(Path(data_dir) / "positives_splitB.csv").copy()
    positive_test_df["sequence_id"] = positive_test_df.get("sequence_id", positive_test_df["accession"]).astype(str)
    positive_test_df["epitope_label"] = [
        parse_epitope_label(seq, start, end)
        for seq, start, end in zip(
            positive_test_df["sequence"],
            positive_test_df["epitope_start"],
            positive_test_df["epitope_end"],
        )
    ]
    positive_test_df["seq_len"] = positive_test_df["sequence"].str.len().astype(int)
    positive_test_df["n_epitope_residues"] = positive_test_df["epitope_label"].map(lambda arr: int(np.asarray(arr).sum()))
    positive_test_df["epitope_density"] = positive_test_df["n_epitope_residues"] / positive_test_df["seq_len"]
    sequence_lookup_df = positive_test_df[["sequence_id", "accession", "sequence", "seq_len", "epitope_density"]].drop_duplicates()
    sequence_lookup = sequence_lookup_df.set_index("sequence_id").to_dict(orient="index")
    return sequence_lookup_df.reset_index(drop=True), sequence_lookup


def parse_score_array(value):
    if isinstance(value, np.ndarray):
        return value.astype(np.float32)
    if isinstance(value, list):
        return np.asarray(value, dtype=np.float32)
    if pd.isna(value):
        raise ValueError("Encountered missing IG score payload.")
    parsed = value
    if isinstance(value, str):
        try:
            parsed = json.loads(value)
        except json.JSONDecodeError:
            parsed = ast.literal_eval(value)
    arr = np.asarray(parsed, dtype=np.float32)
    if arr.ndim != 1:
        raise ValueError(f"Expected 1D IG score array, got shape {arr.shape}.")
    return arr


def load_raw_ig_scores_by_id(raw_ig_rows_path: Path):
    raw_ig_df = pd.read_csv(raw_ig_rows_path)
    raw_ig_score_column = next((col for col in RAW_IG_SCORE_COLUMN_CANDIDATES if col in raw_ig_df.columns), None)
    if raw_ig_score_column is None:
        raise ValueError(
            "The baseline IG artifact does not contain raw per-residue IG scores. "
            f"Checked {raw_ig_rows_path} for columns {list(RAW_IG_SCORE_COLUMN_CANDIDATES)}."
        )
    raw_id_column = "sequence_id" if "sequence_id" in raw_ig_df.columns else "accession"
    ig_method_mask = raw_ig_df["method"].astype(str).str.lower().eq("integrated_gradients") if "method" in raw_ig_df.columns else pd.Series(True, index=raw_ig_df.index)
    return {
        str(row[raw_id_column]): parse_score_array(row[raw_ig_score_column])
        for _, row in raw_ig_df.loc[ig_method_mask].iterrows()
    }


def forward_prob(model, tokenizer, sequence: str, device: str) -> float:
    encodings = tokenizer(sequence, add_special_tokens=False, padding=False, truncation=False, return_tensors="pt")
    encodings = {key: value.to(device) for key, value in encodings.items()}
    with torch.no_grad():
        outputs = model(encodings["input_ids"], encodings["attention_mask"])
    return float(torch.sigmoid(outputs["logits"]).item())


def compute_ig_masking_sweep(model, tokenizer, sequence_lookup_df, raw_ig_scores_by_id, device, confidence_threshold=None, progress_print_every=0):
    validation_rows = []
    p_base_by_sequence = {}
    missing_ig_sequence_ids = []
    included_sequence_ids = set()
    total_rows = int(len(sequence_lookup_df))
    for idx, row in enumerate(sequence_lookup_df.itertuples(index=False), start=1):
        sequence_id = str(row.sequence_id)
        accession = str(row.accession)
        sequence = str(row.sequence)
        p_base = forward_prob(model, tokenizer, sequence, device)
        p_base_by_sequence[sequence_id] = p_base
        if confidence_threshold is not None and p_base <= confidence_threshold:
            continue
        ig_scores = raw_ig_scores_by_id.get(sequence_id)
        if ig_scores is None:
            ig_scores = raw_ig_scores_by_id.get(accession)
        if ig_scores is None:
            missing_ig_sequence_ids.append(sequence_id)
            continue
        if len(ig_scores) != len(sequence):
            raise ValueError(
                f"IG score length mismatch for {sequence_id}: len(scores)={len(ig_scores)} vs len(sequence)={len(sequence)}"
            )
        for k_pct in DEFAULT_K_PERCENTAGES:
            result = validate_ig_residues_by_masking(
                model=model,
                tokenizer=tokenizer,
                sequence=sequence,
                ig_scores=ig_scores,
                device=device,
                k_pct=float(k_pct),
            )
            validation_rows.append({
                "sequence_id": sequence_id,
                "k_pct": float(result["k_pct"]),
                "k_absolute": int(result["k_absolute"]),
                "p_base": float(result["p_base"]),
                "p_masked": float(result["p_masked"]),
                "delta_p": float(result["delta_p"]),
                "validated": bool(result["delta_p"] > DEFAULT_MIN_DELTA_P),
                "top_k_indices_json": json.dumps(result["top_k_indices"]),
            })
        included_sequence_ids.add(sequence_id)
        if progress_print_every > 0 and (idx % progress_print_every == 0 or idx == total_rows):
            print(f"IG masking sweep progress: {idx}/{total_rows} proteins processed; included={len(included_sequence_ids)}")
    sweep_df = pd.DataFrame(validation_rows)
    audit = {
        "n_total_splitb_positive_proteins": int(len(sequence_lookup_df)),
        "n_proteins_passing_default_confidence_filter": int(sum(1 for value in p_base_by_sequence.values() if value > DEFAULT_CONFIDENCE_THRESHOLD)),
        "n_included_proteins": int(sweep_df["sequence_id"].nunique()) if not sweep_df.empty else 0,
        "missing_ig_sequence_ids": missing_ig_sequence_ids,
    }
    return sweep_df, audit


def compute_random_masking_baseline_sweep(model, tokenizer, sequence_lookup, ig_validation_sweep_df, device, progress_print_every=0):
    baseline_rows = []
    total_rows = int(len(ig_validation_sweep_df))
    for idx, row in enumerate(ig_validation_sweep_df.itertuples(index=False), start=1):
        sequence_id = str(row.sequence_id)
        sequence = str(sequence_lookup[sequence_id]["sequence"])
        k_absolute = int(row.k_absolute)
        p_base = float(row.p_base)
        random_delta_ps = []
        for trial_idx in range(DEFAULT_N_RANDOM_TRIALS):
            rng = np.random.default_rng(trial_idx)
            random_indices = rng.choice(len(sequence), size=k_absolute, replace=False)
            masked_residues = list(sequence)
            for residue_idx in random_indices:
                masked_residues[int(residue_idx)] = tokenizer.mask_token
            p_masked_random = forward_prob(model, tokenizer, "".join(masked_residues), device)
            random_delta_ps.append(p_base - p_masked_random)
        mean_random_delta_p = float(np.mean(random_delta_ps))
        baseline_rows.append({
            "sequence_id": sequence_id,
            "k_pct": float(row.k_pct),
            "k_absolute": k_absolute,
            "p_base": p_base,
            "ig_delta_p": float(row.delta_p),
            "mean_random_delta_p": mean_random_delta_p,
            "delta_difference": float(row.delta_p) - mean_random_delta_p,
        })
        if progress_print_every > 0 and (idx % progress_print_every == 0 or idx == total_rows):
            print(f"Random masking sweep progress: {idx}/{total_rows} protein-k rows processed")
    return pd.DataFrame(baseline_rows)


def build_selected_k_random_baseline(random_baseline_sweep_df):
    selected_df = random_baseline_sweep_df.loc[np.isclose(random_baseline_sweep_df["k_pct"].to_numpy(dtype=float), DEFAULT_SELECTED_K_PCT)].copy()
    return selected_df[["sequence_id", "ig_delta_p", "mean_random_delta_p", "delta_difference", "p_base"]].reset_index(drop=True)


def summarize_ig_masking_vs_random(ig_validation_sweep_df, random_baseline_sweep_df):
    summary_rows = []
    for k_pct, group_df in ig_validation_sweep_df.groupby("k_pct", sort=True):
        random_group_df = random_baseline_sweep_df.loc[np.isclose(random_baseline_sweep_df["k_pct"].to_numpy(dtype=float), float(k_pct))].copy()
        ig_mean, ig_ci_low, ig_ci_high = bootstrap_mean_ci(group_df["delta_p"])
        random_mean, random_ci_low, random_ci_high = bootstrap_mean_ci(random_group_df["mean_random_delta_p"])
        validated_fraction = float(group_df["validated"].astype(float).mean())
        summary_rows.append({
            "k_percentage": float(k_pct),
            "n_proteins": int(group_df["sequence_id"].nunique()),
            "ig_mean_delta_p": float(ig_mean),
            "ig_ci_low": float(ig_ci_low),
            "ig_ci_high": float(ig_ci_high),
            "random_mean_delta_p": float(random_mean),
            "random_ci_low": float(random_ci_low),
            "random_ci_high": float(random_ci_high),
            "validated_count": int(group_df["validated"].sum()),
            "validated_fraction": validated_fraction,
        })
    summary_df = pd.DataFrame(summary_rows).sort_values("k_percentage").reset_index(drop=True)
    paired_df = random_baseline_sweep_df.loc[np.isclose(random_baseline_sweep_df["k_pct"].to_numpy(dtype=float), DEFAULT_SELECTED_K_PCT)].dropna(subset=["ig_delta_p", "mean_random_delta_p"]).copy()
    wilcoxon_result = wilcoxon(
        paired_df["ig_delta_p"].to_numpy(dtype=float),
        paired_df["mean_random_delta_p"].to_numpy(dtype=float),
        alternative="two-sided",
    )
    selected_summary = {
        "confidence_filter": "none",
        "n_proteins": int(len(paired_df)),
        "selected_k": float(DEFAULT_SELECTED_K_PCT),
        "ig_mean_delta_p_at_selected_k": float(paired_df["ig_delta_p"].mean()),
        "random_mean_delta_p_at_selected_k": float(paired_df["mean_random_delta_p"].mean()),
        "mean_delta_p_difference_at_selected_k": float((paired_df["ig_delta_p"] - paired_df["mean_random_delta_p"]).mean()),
        "paired_wilcoxon_p_at_selected_k": float(wilcoxon_result.pvalue),
    }
    return summary_df, selected_summary


def significance_marker(p_value: float) -> str:
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    return "ns"


def plot_supplementary_ig_masking_vs_random_unfiltered(summary_df, selected_summary, pdf_path: Path, png_path: Path):
    fig, ax = plt.subplots(figsize=(3.35, 2.8))
    ax.plot(summary_df["k_percentage"], summary_df["ig_mean_delta_p"], marker="o", markersize=3.8, linewidth=1.4, color="#4C72B0", label="IG-guided masking")
    ax.fill_between(summary_df["k_percentage"], summary_df["ig_ci_low"], summary_df["ig_ci_high"], color="#4C72B0", alpha=0.18)
    ax.plot(summary_df["k_percentage"], summary_df["random_mean_delta_p"], marker="D", markersize=3.3, linewidth=1.3, linestyle="--", color="#C44E52", label="Random masking")
    ax.fill_between(summary_df["k_percentage"], summary_df["random_ci_low"], summary_df["random_ci_high"], color="#C44E52", alpha=0.14)
    selected_row = summary_df.loc[np.isclose(summary_df["k_percentage"].to_numpy(dtype=float), DEFAULT_SELECTED_K_PCT)].iloc[0]
    ig_selected_mean = float(selected_row["ig_mean_delta_p"])
    ig_selected_ci_high = float(selected_row["ig_ci_high"])
    random_selected_mean = float(selected_row["random_mean_delta_p"])
    random_selected_ci_high = float(selected_row["random_ci_high"])
    ax.errorbar([DEFAULT_SELECTED_K_PCT], [random_selected_mean], yerr=[[random_selected_mean - float(selected_row["random_ci_low"])], [float(selected_row["random_ci_high"]) - random_selected_mean]], fmt="D", markersize=4.5, capsize=3, color="#C44E52")
    ax.errorbar([DEFAULT_SELECTED_K_PCT], [ig_selected_mean], yerr=[[ig_selected_mean - float(selected_row["ig_ci_low"])], [ig_selected_ci_high - ig_selected_mean]], fmt="none", capsize=3, ecolor="#4C72B0", elinewidth=1.4)
    ax.scatter([DEFAULT_SELECTED_K_PCT], [ig_selected_mean], s=44, facecolors="white", edgecolors="#4C72B0", linewidths=1.6, zorder=5)
    y_top = float(max(summary_df["ig_ci_high"].max(), summary_df["random_ci_high"].max()))
    y_bottom = float(min(summary_df["ig_ci_low"].min(), summary_df["random_ci_low"].min(), random_selected_mean, ig_selected_mean))
    y_range = max(y_top - y_bottom, 1e-6)
    bracket_x = DEFAULT_SELECTED_K_PCT + 0.022
    tick_width = 0.012
    ax.plot([bracket_x, bracket_x + tick_width, bracket_x + tick_width, bracket_x], [ig_selected_mean, ig_selected_mean, random_selected_mean, random_selected_mean], color="black", linewidth=1.1)
    p_value = float(selected_summary["paired_wilcoxon_p_at_selected_k"])
    ax.text(bracket_x + tick_width * 0.5, max(ig_selected_ci_high, random_selected_ci_high) + 0.03 * y_range, significance_marker(p_value), ha="center", va="bottom", fontsize=8)
    ax.set_xlabel("Top-k percentage")
    ax.set_ylabel("Mean Δp")
    ax.set_xlim(float(summary_df["k_percentage"].min()) - 0.025, float(summary_df["k_percentage"].max()) + 0.025)
    ax.set_ylim(y_bottom - 0.04 * y_range, y_top + 0.12 * y_range)
    ax.legend(frameon=False, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.40), ncol=2)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.tick_params(labelsize=7)
    fig.tight_layout(rect=(0.0, 0.18, 1.0, 1.0))
    pdf_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


In [7]:
if RUN_TARGET == "colab":
    PROJECT_ROOT = DRIVE_ROOT if DRIVE_ROOT.exists() else RUNTIME_ROOT
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Project root: {PROJECT_ROOT}")
    print(f"Runtime root: {RUNTIME_ROOT}")
    print(f"Device: {DEVICE}")
    if DEVICE == "cuda":
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    DEVICE = detect_device()
    print_runtime_context(DEVICE, PROJECT_ROOT)

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
ISM_RESULTS_DIR = RESULTS_DIR / "insilico_mutagenesis"
PAPER_FIGURES_DIR = RESULTS_DIR / "paper_figures"
PAPER_TABLES_DIR = RESULTS_DIR / "paper_tables"
for _d in [ISM_RESULTS_DIR, PAPER_FIGURES_DIR, PAPER_TABLES_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = MODELS_DIR / "baseline_frozen_esm2.pt"
RAW_IG_ROWS_PATH = RESULTS_DIR / "probing" / "rows" / "baseline_probing_rows.csv"
model, _ = load_baseline_checkpoint(CHECKPOINT_PATH, DEVICE)
tokenizer = build_tokenizer()
sequence_lookup_df, sequence_lookup = load_splitb_positive_sequence_lookup(DATA_DIR)
raw_ig_scores_by_id = load_raw_ig_scores_by_id(RAW_IG_ROWS_PATH)

print(f"Using checkpoint: {CHECKPOINT_PATH}")
print(f"Using raw IG artifact: {RAW_IG_ROWS_PATH}")
print(f"Loaded splitB positives with epitope masks: {len(sequence_lookup_df)} proteins")


Project root: /content/drive/MyDrive/XAllergen2.0
Runtime root: /content/XAllergen2.0
Device: cuda
  GPU: NVIDIA A100-SXM4-40GB
  VRAM: 42.4 GB


Some weights of EsmModel were not initialized from the model checkpoint at /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using checkpoint: /content/drive/MyDrive/XAllergen2.0/models/baseline_frozen_esm2.pt
Using raw IG artifact: /content/drive/MyDrive/XAllergen2.0/results/probing/rows/baseline_probing_rows.csv
Loaded splitB positives with epitope masks: 61 proteins


In [8]:
unfiltered_sweep_df, audit = compute_ig_masking_sweep(
    model,
    tokenizer,
    sequence_lookup_df,
    raw_ig_scores_by_id,
    DEVICE,
    confidence_threshold=None,
    progress_print_every=5,
)

filtered_n = int(audit["n_proteins_passing_default_confidence_filter"])
unfiltered_n = int(audit["n_included_proteins"])
print(
    "IG masking audit: "
    f"filtered n (F(x) > {DEFAULT_CONFIDENCE_THRESHOLD:.2f}) = {filtered_n}; "
    f"unfiltered n = {unfiltered_n}"
)
if audit["missing_ig_sequence_ids"]:
    print("Missing IG scores for:", ", ".join(str(v) for v in audit["missing_ig_sequence_ids"]))


IG masking sweep progress: 5/61 proteins processed; included=5
IG masking sweep progress: 10/61 proteins processed; included=10
IG masking sweep progress: 15/61 proteins processed; included=15
IG masking sweep progress: 20/61 proteins processed; included=20
IG masking sweep progress: 25/61 proteins processed; included=25
IG masking sweep progress: 30/61 proteins processed; included=30
IG masking sweep progress: 35/61 proteins processed; included=35
IG masking sweep progress: 40/61 proteins processed; included=40
IG masking sweep progress: 45/61 proteins processed; included=45
IG masking sweep progress: 50/61 proteins processed; included=50
IG masking sweep progress: 55/61 proteins processed; included=55
IG masking sweep progress: 60/61 proteins processed; included=60
IG masking sweep progress: 61/61 proteins processed; included=61
IG masking audit: filtered n (F(x) > 0.70) = 46; unfiltered n = 61


In [9]:
random_sweep_df = compute_random_masking_baseline_sweep(
    model,
    tokenizer,
    sequence_lookup,
    unfiltered_sweep_df,
    DEVICE,
    progress_print_every=25,
)
selected_random_df = build_selected_k_random_baseline(random_sweep_df)
sweep_summary_df, selected_summary = summarize_ig_masking_vs_random(unfiltered_sweep_df, random_sweep_df)

unfiltered_raw_sweep_csv = ISM_RESULTS_DIR / "ig_validation_sweep_unfiltered.csv"
unfiltered_random_sweep_csv = ISM_RESULTS_DIR / "ig_vs_random_baseline_sweep_unfiltered.csv"
unfiltered_selected_random_csv = ISM_RESULTS_DIR / "ig_vs_random_baseline_unfiltered.csv"
unfiltered_summary_csv = ISM_RESULTS_DIR / "ig_masking_sweep_unfiltered.csv"
unfiltered_sweep_df.to_csv(unfiltered_raw_sweep_csv, index=False)
random_sweep_df.to_csv(unfiltered_random_sweep_csv, index=False)
selected_random_df.to_csv(unfiltered_selected_random_csv, index=False)
sweep_summary_df.to_csv(unfiltered_summary_csv, index=False)

summary_csv = PAPER_TABLES_DIR / "supplementary_ig_masking_vs_random_unfiltered_summary.csv"
summary_tex = PAPER_TABLES_DIR / "supplementary_ig_masking_vs_random_unfiltered_summary.tex"
summary_json = PAPER_TABLES_DIR / "supplementary_ig_masking_vs_random_unfiltered_summary.json"
summary_df = pd.DataFrame([selected_summary])
summary_df.to_csv(summary_csv, index=False)
summary_tex.write_text(summary_df.to_latex(index=False, escape=False), encoding="utf-8")
summary_json.write_text(json.dumps(selected_summary, indent=2), encoding="utf-8")

print(f"Saved raw unfiltered sweep to: {unfiltered_raw_sweep_csv}")
print(f"Saved unfiltered random sweep to: {unfiltered_random_sweep_csv}")
print(f"Saved selected-k unfiltered random baseline to: {unfiltered_selected_random_csv}")
print(f"Saved unfiltered sweep summary to: {unfiltered_summary_csv}")
summary_df


Random masking sweep progress: 25/549 protein-k rows processed
Random masking sweep progress: 50/549 protein-k rows processed
Random masking sweep progress: 75/549 protein-k rows processed
Random masking sweep progress: 100/549 protein-k rows processed
Random masking sweep progress: 125/549 protein-k rows processed
Random masking sweep progress: 150/549 protein-k rows processed
Random masking sweep progress: 175/549 protein-k rows processed
Random masking sweep progress: 200/549 protein-k rows processed
Random masking sweep progress: 225/549 protein-k rows processed
Random masking sweep progress: 250/549 protein-k rows processed
Random masking sweep progress: 275/549 protein-k rows processed
Random masking sweep progress: 300/549 protein-k rows processed
Random masking sweep progress: 325/549 protein-k rows processed
Random masking sweep progress: 350/549 protein-k rows processed
Random masking sweep progress: 375/549 protein-k rows processed
Random masking sweep progress: 400/549 prot

,confidence_filter,n_proteins,selected_k,ig_mean_delta_p_at_selected_k,random_mean_delta_p_at_selected_k,mean_delta_p_difference_at_selected_k,paired_wilcoxon_p_at_selected_k
0,none,61,0.4,0.237576,-0.022653,0.260228,3.738478e-07


In [10]:
fig_pdf = PAPER_FIGURES_DIR / "supplementary_ig_masking_vs_random_unfiltered.pdf"
fig_png = PAPER_FIGURES_DIR / "supplementary_ig_masking_vs_random_unfiltered.png"
plot_supplementary_ig_masking_vs_random_unfiltered(sweep_summary_df, selected_summary, fig_pdf, fig_png)

print(f"Saved supplementary figure to: {fig_pdf}")
print(f"Saved supplementary figure to: {fig_png}")
print(
    "Selected k summary: "
    f"n={selected_summary['n_proteins']}, "
    f"IG mean Δp={selected_summary['ig_mean_delta_p_at_selected_k']:.6f}, "
    f"random mean Δp={selected_summary['random_mean_delta_p_at_selected_k']:.6f}, "
    f"mean diff={selected_summary['mean_delta_p_difference_at_selected_k']:.6f}, "
    f"paired Wilcoxon p={selected_summary['paired_wilcoxon_p_at_selected_k']:.3e}"
)
sweep_summary_df


Saved supplementary figure to: /content/drive/MyDrive/XAllergen2.0/results/paper_figures/supplementary_ig_masking_vs_random_unfiltered.pdf
Saved supplementary figure to: /content/drive/MyDrive/XAllergen2.0/results/paper_figures/supplementary_ig_masking_vs_random_unfiltered.png
Selected k summary: n=61, IG mean Δp=0.237576, random mean Δp=-0.022653, mean diff=0.260228, paired Wilcoxon p=3.738e-07


,k_percentage,n_proteins,ig_mean_delta_p,ig_ci_low,ig_ci_high,random_mean_delta_p,random_ci_low,random_ci_high,validated_count,validated_fraction
0,0.05,61,0.038554,0.010562,0.069395,0.010051,0.005686,0.014870,16,0.262295
1,0.10,61,0.082703,0.042006,0.127127,0.016629,0.009014,0.025054,23,0.377049
2,0.20,61,0.115872,0.061949,0.170720,0.023719,0.008570,0.038170,34,0.557377
3,0.25,61,0.151905,0.093370,0.212139,0.026685,0.007224,0.045883,38,0.622951
4,0.30,61,0.210048,0.134032,0.281432,0.020239,-0.004792,0.043595,45,0.737705
5,0.35,61,0.263681,0.172504,0.349216,0.000458,-0.032319,0.029466,41,0.672131
6,0.40,61,0.237576,0.142219,0.327525,-0.022653,-0.064485,0.013563,41,0.672131
7,0.45,61,0.198730,0.102385,0.291625,-0.052256,-0.108767,-0.003120,37,0.606557
8,0.50,61,0.063859,-0.028938,0.149896,-0.078453,-0.150967,-0.015123,32,0.524590


In [15]:
unfiltered_summary_csv = ISM_RESULTS_DIR / "ig_masking_sweep_unfiltered.csv"
summary_csv = PAPER_TABLES_DIR / "supplementary_ig_masking_vs_random_unfiltered_summary.csv"

sweep_summary_df = pd.read_csv(unfiltered_summary_csv)
selected_summary = pd.read_csv(summary_csv).iloc[0].to_dict()

fig_pdf = PAPER_FIGURES_DIR / "supplementary_ig_masking_vs_random_unfiltered.pdf"
fig_png = PAPER_FIGURES_DIR / "supplementary_ig_masking_vs_random_unfiltered.png"

plot_supplementary_ig_masking_vs_random_unfiltered(
    sweep_summary_df,
    selected_summary,
    fig_pdf,
    fig_png,
)

print(f"Regenerated: {fig_pdf}")
print(f"Regenerated: {fig_png}")
sweep_summary_df


Regenerated: /content/drive/MyDrive/XAllergen2.0/results/paper_figures/supplementary_ig_masking_vs_random_unfiltered.pdf
Regenerated: /content/drive/MyDrive/XAllergen2.0/results/paper_figures/supplementary_ig_masking_vs_random_unfiltered.png


,k_percentage,n_proteins,ig_mean_delta_p,ig_ci_low,ig_ci_high,random_mean_delta_p,random_ci_low,random_ci_high,validated_count,validated_fraction
0,0.05,61,0.038554,0.010562,0.069395,0.010051,0.005686,0.014870,16,0.262295
1,0.10,61,0.082703,0.042006,0.127127,0.016629,0.009014,0.025054,23,0.377049
2,0.20,61,0.115872,0.061949,0.170720,0.023719,0.008570,0.038170,34,0.557377
3,0.25,61,0.151905,0.093370,0.212139,0.026685,0.007224,0.045883,38,0.622951
4,0.30,61,0.210048,0.134032,0.281432,0.020239,-0.004792,0.043595,45,0.737705
5,0.35,61,0.263681,0.172504,0.349216,0.000458,-0.032319,0.029466,41,0.672131
6,0.40,61,0.237576,0.142219,0.327525,-0.022653,-0.064485,0.013563,41,0.672131
7,0.45,61,0.198730,0.102385,0.291625,-0.052256,-0.108767,-0.003120,37,0.606557
8,0.50,61,0.063859,-0.028938,0.149896,-0.078453,-0.150967,-0.015123,32,0.524590
